In [ ]:
!pip install -q torch transformers peft bitsandbytes accelerate datasets trl wandb

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')  # auth click, one-time per session


In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train":"train.jsonl","validation":"val.jsonl","test":"test.jsonl"})
print(ds)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, re
from collections import Counter

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tok = AutoTokenizer.from_pretrained(model_id); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(model_id,
  quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True), device_map="auto")

def f1(pred, ref):
    pt, rt = Counter(re.findall(r"[a-z0-9]+", pred.lower())), Counter(re.findall(r"[a-z0-9]+", ref.lower()))
    ov = sum((pt & rt).values()); 
    if not ov: return 0.0
    p, r = ov/sum(pt.values()), ov/sum(rt.values())
    return 2*p*r/(p+r)

scores = []
for ex in ds["test"]:
    prompt = f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n"
    ids = tok(prompt, return_tensors="pt").to(base.device)
    out = base.generate(**ids, max_new_tokens=128, do_sample=False)
    pred = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    scores.append(f1(pred, ex["output"]))
print(f"BASELINE mean F1 on test-100: {sum(scores)/len(scores):.3f}")

In [ ]:
# Cell A — reload base once (weights cached now, ~1 min)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
tok = AutoTokenizer.from_pretrained(model_id); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(model_id,
  quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True), device_map="auto")

In [ ]:
# Cell B — clean text datasets + proof train
def to_text(e):
    return {"text": f"### Instruction:\n{e['instruction']}\n\n### Input:\n{e['input']}\n\n### Response:\n{e['output']}"}
train_ds = ds["train"].map(to_text, remove_columns=ds["train"].column_names)
val_ds = ds["validation"].map(to_text, remove_columns=ds["validation"].column_names)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
base = prepare_model_for_kbit_training(base)
model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
args = SFTConfig(output_dir="adapters/legal-qlora", num_train_epochs=3,
  per_device_train_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4,
  max_length=1024, eval_strategy="steps", eval_steps=50, logging_steps=10,
  report_to="none")
SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=args).train()

In [ ]:
print(list(model.peft_config.keys()))  # stacking verdict, still owed

In [ ]:
import os
print(os.listdir('adapters/legal-qlora'))
import shutil
shutil.make_archive('/content/drive/MyDrive/d3/adapter-final150', 'zip', root_dir='adapters/legal-qlora')
print([f for f in os.listdir('/content/drive/MyDrive/d3') if 'adapter' in f])

In [ ]:
from peft import PeftModel
tuned = PeftModel.from_pretrained(base, "adapters/legal-qlora/checkpoint-150")
tuned.eval()

import re
from collections import Counter
def f1(pred, ref):
    pt, rt = Counter(re.findall(r"[a-z0-9]+", pred.lower())), Counter(re.findall(r"[a-z0-9]+", ref.lower()))
    ov = sum((pt & rt).values())
    if not ov: return 0.0
    p, r = ov/sum(pt.values()), ov/sum(rt.values())
    return 2*p*r/(p+r)

scores = []
for ex in ds["test"]:
    prompt = f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n"
    ids = tok(prompt, return_tensors="pt").to(tuned.device)
    out = tuned.generate(**ids, max_new_tokens=128, do_sample=False)
    pred = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    scores.append(f1(pred, ex["output"]))
print(f"AFTER (ckpt-150) mean F1 on test-100: {sum(scores)/len(scores):.3f}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
fresh = AutoModelForCausalLM.from_pretrained(model_id,
  quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
  bnb_4bit_use_double_quant=True), device_map="auto")
tuned2 = PeftModel.from_pretrained(fresh, "adapters/legal-qlora/checkpoint-150")
print(list(tuned2.peft_config.keys()))

In [ ]:
import re
from collections import Counter
def f1(pred, ref):
    pt, rt = Counter(re.findall(r"[a-z0-9]+", pred.lower())), Counter(re.findall(r"[a-z0-9]+", ref.lower()))
    ov = sum((pt & rt).values())
    if not ov: return 0.0
    p, r = ov/sum(pt.values()), ov/sum(rt.values())
    return 2*p*r/(p+r)
scores = []
for ex in ds["test"]:
    prompt = f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n"
    ids = tok(prompt, return_tensors="pt").to(tuned2.device)
    out = tuned2.generate(**ids, max_new_tokens=128, do_sample=False)
    pred = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    scores.append(f1(pred, ex["output"]))
print(f"AFTER clean mean F1 on test-100: {sum(scores)/len(scores):.3f}")